# Apply the status rules to seven situations

*Adversarial Mathematics From First Principles*, [chapter 24](https://programmer.ie/books/math/24-chapter/) · [worked example](https://programmer.ie/examples/math/24-chapter/)

Run the cells in order. The program is a small, self-contained demonstration of the chapter's mathematics; the output saved with it is what it printed when this notebook was built.

Two questions have different answerers: did the checks pass (the machine)
and does the formal statement say what the claim means (a person). The runner
decides a status by a fixed order of rules, and counts an approval only if it
is from a human role, approves, names the right correspondence and the exact
contract hash. This program applies those rules to seven situations.

## Run it

Standard library only; nothing to install.

In [1]:
"""Statuses and approvals: the runner's rules, applied to six situations (standard library only).

The order of the rules and the four conditions on an approval are those of
`math-engine proof`; the situations are made up to exercise them.
"""
APPROVING_ROLES = {"user", "human"}
CONTRACT_SHA = "ef384bbf…"            # the contract a run used (shortened)
CORR = "N1-CORR-01"


def valid_approvals(records):
    return [a for a in records
            if a.get("reviewer_role") in APPROVING_ROLES
            and a.get("decision") == "approve"
            and a.get("correspondence") == CORR
            and a.get("contract_sha256") == CONTRACT_SHA]


def status(all_passed, all_systems, pinned, records, prefix="kernel_checked"):
    if not all_passed:
        return "run_failed"
    if not all_systems:
        return "partial_run"
    if not pinned:
        return "checked_unpinned"
    return f"{prefix}_correspondence_{'approved' if valid_approvals(records) else 'pending'}"


author = {"reviewer_role": "user", "decision": "approve", "correspondence": CORR, "contract_sha256": CONTRACT_SHA}
cases = [
    ("Fermat, before approval", True, True, True, []),
    ("Fermat, after the author approves", True, True, True, [author]),
    ("approval written by an AI assistant", True, True, True, [{**author, "reviewer_role": "assistant"}]),
    ("approval of an older contract", True, True, True, [{**author, "contract_sha256": "0b7c41…"}]),
    ("approved, but a control failed", False, True, True, [author]),
    ("approved, but rehearsed on other tools", True, True, False, [author]),
    ("approved, but only Lean was run", True, False, True, [author]),
]
for name, passed, systems, pinned, records in cases:
    print(f"{name:40} -> {status(passed, systems, pinned, records)}")
print("\nThe first rule that applies wins: an approval never outweighs a failure or upgrades a rehearsal.")

Fermat, before approval                  -> kernel_checked_correspondence_pending
Fermat, after the author approves        -> kernel_checked_correspondence_approved
approval written by an AI assistant      -> kernel_checked_correspondence_pending
approval of an older contract            -> kernel_checked_correspondence_pending
approved, but a control failed           -> run_failed
approved, but rehearsed on other tools   -> checked_unpinned
approved, but only Lean was run          -> partial_run

The first rule that applies wins: an approval never outweighs a failure or upgrades a rehearsal.


## What the output shows

Fermat moves from pending to approved when the author approves. An approval
written by an AI assistant, or recorded against an older contract, is ignored.
A failed control gives `run_failed` even with an approval; a rehearsal on
other tools stays `checked_unpinned`; a partial run is `partial_run`. The
first rule that applies wins.

## Not shown

- The rules and conditions are those of `math-engine proof`; the situations are constructed to exercise them.
- An approval records that the correspondence was accepted by the author; it is not a mathematical review.

## Challenge it

Examine the status rules and the four conditions on an approval. Construct a
situation the seven cases miss, and say which status it should get. Argue for
or against the rule that an AI's approval never counts, and explain why the
order of the rules matters.